# Стохастические методы оптимизации

По большому счету, все методы оптимизации можно разделить на два больших класса: детерминированные и стохастические.

Разница между ними заключается в том, что в детерминированных методах на каждом шаге точка, в которую мы перейдем на следующем шаге, вычисляется однозначно, по формуле или алгоритму, а в стохастических методах выбирается случайно каким-либо образом.

В первой работе мы с вами познакомились с обоими подходами: метод покоординатного спуска - классический представитель детерминированных методов, а метод случайного поиска, как следует из его названия - представитель класса стохастических методов.

Казалось бы, детерминированные методы предпочтительнее, ведь они всегда выбирают "наилучшее" решение на каждом шаге?

К сожалению, они идеально подходят только для задач с гладкими монотонно возрастающими или убывающими целевыми функциями. Если целевая функция имеет локальные минимумы или максимумы, есть большая вероятность, что алгоритм оптимизации остановится, достигнув именно локального максимума или минимума, но не найдет глобальный.

Для задач, где целевая функция имеет сложную форму, стохастические методы зачастую оказываются более подходящими.

Одним из типичных представителей стохастических методов является *метод отжига*.

## Метод отжига

Этот метод был разработан в 1953, причем основа метода пришла из обработки металлов.

В то время перед металлургами стояла задача получения качественных, прочных металлических изделий. На тот момент было известно, что прочность, среди прочих факторов, определяется также структурой металла, а именно, образовали ли молекулы металла кристаллическую решетку в процессе остывания из расплава.

Понятно, что в жидком металле кристаллическая решетка отсутствует, и молекулы могу свободно перемещаться, а в застывшем металле молекулы перемещаться не могут, но от того, встали ли они на "правильное" место, на свое место в кристаллической решетке, зависит прочность готового изделия.

В общем-то, молекулы металла стремятся занять свои места в решетке, поскольку именно в этом случае энергия системы будет наименьшей (собственно, именнно поэтому и прочность изделия в этом случае выше, так как в металле отстутствуют точки напряжения), но не всегда "успевают" это сделать в процессе остывания металла.

Занять свои места они могут, свободно двигаясь в расплаве (при этом с большей вероятностью молекула перейдет в  точку, соответствующую месту в кристаллической решетке), но по мере того, как температура расплава уменьшается, подвижность молекул металла падает. Поэтому металлурги повышали качество металлических отливок, добиваясь *постепенного* и *плавного* уменьшения температуры, чтобы все (или хотя бы большинство) молекул металла успевали занять свои места в кристаллической структуре.

Собственно, метод отжига в оптимизации заимствует идеи из металлургии.

В нашем случае энергии системы соответствует значение целевой функции, молекулам металла соответствует текущая точка в пространстве состояний системы (текущие значения переменных $x$), температура соответствует некоторой величине, которая определяет, как далеко от текущей точки может находиться следующая, в которую мы перейдем, причем температура падает с каждым шагом. Процесс останавливается, когда температура достигнет нуля.

Таким образом, алгоритм включает в себя следующие элементы:

* текущая точка $x$
* значение целевой функции в этой точке $f(x)$
* текущее значение "температуры" $T$
* функция уменьшения температуры на каждом шаге
* функция выбора следующей точки $x^n = G(x)$
* функция вычисления вероятности перехода в следующую точку в зависимости от разницы значений целевой функции в новой и старой точках и текущей температуры $h(f(x^n) - f(x), T)$

Алгоритм состоит из следующих шагов:

1. Выбрать начальную точку $x$, начальную температуру $T$, задать границу вероятности перехода в новую точку $P$
2. Пока температура $T$ больше нуля, повторять:
    1. Вычислить значение целевой функции $f(x)$ в текущей точке $x$
    2. Случайным образом выбрать смещение от текущей точки $x$, и получить новую точку $x^n$
    3. Вычислить значение целевой функции в новой точке $f(x^n)$
    4. Вычислить вероятность перехода в новую точку $h = h(f(x^n) - f(x), T)$
    5. Если вычисленная вероятность $h$ больше заданной $P$, перейти в новую точку, иначе повторить шаг 2.B.
    6. понизить темепературу

Алгоритм останавливается, когда значение темепературы достигнет нуля.

Использование алгоритма не гаратирует нахождения глобального оптимума, но сам по себе алгоритм являет довольно эффективным.

Рассмотрим метод отжига на примере:

In [ ]:
import math
import random

# Это сам метод отжига
# в качестве параметров передаются начальное значение x, начальное значение температуры t,
# функция, уменьшающая значение температуры на каждом шаге decrease_temperature, граница вероятности p, минимизируемая функция f,
# функция генерации нового значения x generate_new_x, функция вычисления вероятности перехода в новую точку h
# в качестве результата функция возвращает найденное значение x, при котором функция f минимальна, и само значение этой функции f(x)
def annealing(x: float, temperature: float, decrease_temperature: callable, p: float, f: callable, generate_new_x: callable, h: callable) -> (float, float):
    while temperature > 0:
        target_function_value = f(x)
        print(f"x = {x}, f(x)={target_function_value}")
        while True:
            x_n = generate_new_x(x)
            new_target_function_value = f(x_n)

            new_probability = h(new_target_function_value - target_function_value, temperature)

            if new_probability > p:
                print(f"x = {x}, f(x) = {target_function_value}, x_new = {x_n}, f(x) = {new_target_function_value}, prob = {new_probability}")

                x = x_n
                target_function_value = new_target_function_value
                break
        temperature = decrease_temperature(temperature)

    return x, f(x)

# это функция вычисления веряотности перехода в новую точку
def h_probability_function(delta_e: float, temperature: float) -> float:
    return math.exp(- delta_e / temperature)

# эта функция уменьшает темепературу на каждом шаге
def linear_decrease_temperature(current_temperature: float) -> float:
    return current_temperature - 1

# эта функция вычисляет новую точку на основе текущей (в данном случае случайно)
def generate_new_x(x: float) -> float:
    return x + random.random() - 0.5

# это функция, которую мы минимизируем
def func_to_minimize(x):
    return (x - 2) * (x - 2) + 1

In [ ]:
# найдем минимум, используя метод имитации отжига
# с начальной точкой x = 0, начальной температурой 100 (то есть всего будет 100 шагов), с вероятностью перехода
# в новую точку 0.15
annealing(0, 100, linear_decrease_temperature, (1 - 0.15), func_to_minimize, generate_new_x, h_probability_function)

x = 0, f(x)=5
x = 0, f(x) = 5, x_new = 0.10584311111909483, f(x) = 4.587830319694989, prob = 1.0041302026775152
x = 0.10584311111909483, f(x)=4.587830319694989
x = 0.10584311111909483, f(x) = 4.587830319694989, x_new = -0.29432669408621703, f(x) = 6.263934979196589, prob = 0.9832121628829105
x = -0.29432669408621703, f(x)=6.263934979196589
x = -0.29432669408621703, f(x) = 6.263934979196589, x_new = 0.07170746864142719, f(x) = 4.718312086493253, prob = 1.015896691275044
x = 0.07170746864142719, f(x)=4.718312086493253
x = 0.07170746864142719, f(x) = 4.718312086493253, x_new = -0.09419767420449421, f(x) = 5.3856638986435135, prob = 0.9931436968485214
x = -0.09419767420449421, f(x)=5.3856638986435135
x = -0.09419767420449421, f(x) = 5.3856638986435135, x_new = 0.07770262505749403, f(x) = 4.6952271977108495, prob = 1.0072179738660059
x = 0.07770262505749403, f(x)=4.6952271977108495
x = 0.07770262505749403, f(x) = 4.6952271977108495, x_new = -0.40315129076090506, f(x) = 6.775136126285804, pr

(1.9018239574998494, 1.0096385353209913)

Выше рассмотрен пример для поиска минимума функции, которую мы уже рассматривали, $f(x) = (x - 2)^2 + 1$.

Попробуйте запустить предыдущую ячейку несколько раз и убедитесь, что результат каждый раз несколько различается.

Попробуйте несколько различных видов функции уменьшения темепературы (например, увеличьте или уменьшите скорость,  с которой меняется температура, попробуйте нелинейное уменьшение, когда вначале темепература уменьшается быстро, а затем все медленнее и медленнее, или наоборот), оцените эффективность такого подхода.

Список дополнительной литературы:

[Алгоритм имитации отжига](https://ru.wikipedia.org/wiki/Алгоритм_имитации_отжига)

[Введение в оптимизацию. Имитация отжига](https://habr.com/ru/post/209610/)

[Метод отжига](https://www.math.spbu.ru/user/gran/sb1/lopatin.pdf)

Задание для самостоятельной работы:

Используйте целевую функцию из работы №2 (минимизация времени пересечения поля).
Подберите начально значение темепературы таким образом, чтобы поставленная задача была решена (возможно, его придется менять несколько раз)

In [1]:
import math
import random

# --- БЛОК 1: Исходные данные из работы №2 ---
h1 = 0.10
h2 = 0.10
l = 1.0
v1 = 40.0
v2 = 30.0
c1 = 0.115
c2 = 0.15
a1 = 0.5
a2 = 0.5

# --- БЛОК 2: Целевая функция из работы №2 (минимизация) ---
def func_to_minimize(x):
    # Убедимся, что x не выходит за пределы дороги.
    # Если алгоритм предложит x вне [0, l], мы вернем огромное значение,
    # чтобы "наказать" его и заставить искать внутри интервала.
    if x < 0 or x > l:
        return float('inf')

    dist1 = math.sqrt(h1 * h1 + x * x)
    dist2 = math.sqrt(h2 * h2 + (l - x) * (l - x))

    time1 = dist1 / v1
    time2 = dist2 / v2
    total_time = time1 + time2

    fuel1 = dist1 * c1
    fuel2 = dist2 * c2
    total_fuel = fuel1 + fuel2

    return a1 * total_time + a2 * total_fuel

# --- БЛОК 3: Функции для метода отжига (адаптированные) ---

def h_probability_function(delta_e: float, temperature: float) -> float:
    """
    Вычисляет вероятность перехода.
    Чем хуже новая точка (delta_e > 0), тем меньше вероятность.
    Чем выше температура, тем больше вероятность принять плохое решение.
    """
    if temperature <= 0:
        return 0.0
    # Если новая точка лучше (delta_e < 0), то math.exp(положительное число) > 1,
    # и мы гарантированно перейдем, т.к. вероятность будет больше p.
    return math.exp(-delta_e / temperature)

def linear_decrease_temperature(current_temperature: float) -> float:
    """Простое линейное уменьшение температуры."""
    return current_temperature - 1

def generate_new_x(x: float) -> float:
    """
    Генерирует новую точку, делая случайный шаг.
    Важно: шаг должен быть соизмерим с длиной отрезка [0, l].
    """
    # Будем делать шаг не больше 0.1 от длины дороги.
    # random.uniform(-0.1, 0.1) даст случайное смещение в пределах [-0.1, 0.1] км.
    step_size = 0.1
    new_x = x + random.uniform(-step_size, step_size)
    return new_x

# --- БЛОК 4: Функция имитации отжига (немного доработанная) ---
def annealing(x_start: float,
              initial_temperature: float,
              decrease_temperature: callable,
              p: float,
              f: callable,
              generate_new_x: callable,
              h: callable,
              max_iterations: int = 1000) -> (float, float):
    """
    Добавлена защита от бесконечного цикла, если температура не доходит до нуля.
    """
    x = x_start
    temperature = initial_temperature
    iteration = 0

    while temperature > 0 and iteration < max_iterations:
        target_function_value = f(x)

        # Внутренний цикл: пытаемся найти новую точку для перехода
        for _ in range(100):  # Ограничим число попыток на одной температуре
            x_n = generate_new_x(x)
            new_target_function_value = f(x_n)

            # Если новая точка "запретная" (inf), пропускаем её
            if math.isinf(new_target_function_value):
                continue

            delta = new_target_function_value - target_function_value
            new_probability = h(delta, temperature)

            # Переходим, если вероятность выше порога
            if new_probability > p:
                x = x_n
                break  # Выходим из внутреннего цикла, переходим к снижению температуры
        # else: # Этот else выполнится, если break не сработал (не нашли подходящую точку)
            # Можно ничего не делать, просто понизить температуру и продолжить с текущей x.

        temperature = decrease_temperature(temperature)
        iteration += 1

    return x, f(x)

# --- БЛОК 5: Запуск и подбор параметров ---

# 1. Начальная точка. Возьмем середину дороги.
start_x = 0.5

# 2. Вероятность перехода. Оставим как в примере, или возьмем 0.5.
# Суть: мы переходим, если h > p. Для хороших точек h > 1 всегда, так что p не критично.
threshold_p = 0.3

# 3. ГЛАВНЫЙ ПАРАМЕТР: Начальная температура.
# Для нашей функции характерные значения f(x) - около 0.04-0.05.
# Разница между значениями в разных точках (delta) может быть ~0.001-0.01.
# Чтобы exp(-delta/T) был > 0.3 для "плохих" точек, нужно T, сравнимое с delta.
# Если T будет слишком мал, алгоритм быстро застрянет в локальном минимуме (если бы они были).
# Если T будет слишком велик, алгоритм будет долго скакать как пьяный.
# Путем экспериментов я подобрал T=0.01, так как это примерно масштаб характерных изменений функции.
# Попробуйте T = 0.1, 0.01, 0.001 и посмотрите на результат.
initial_temp = 0.01

print(f"Поиск минимума методом имитации отжига")
print(f"Начальная температура: {initial_temp}")
print("-" * 30)

# Запускаем алгоритм несколько раз, чтобы увидеть статистику
num_runs = 5
for run in range(num_runs):
    final_x, final_f = annealing(start_x,
                                 initial_temp,
                                 linear_decrease_temperature,
                                 threshold_p,
                                 func_to_minimize,
                                 generate_new_x,
                                 h_probability_function,
                                 max_iterations=500) # Ограничим число итераций
    print(f"Run {run+1}: x* = {final_x:.4f}, f(x*) = {final_f:.6f}")

print("-" * 30)
# Для сравнения, результат из scipy.optimize (детерминированный метод)
from scipy.optimize import minimize_scalar
result_scipy = minimize_scalar(func_to_minimize, bounds=(0, l), method='bounded')
print(f"Для справки (minimize_scalar): x* = {result_scipy.x:.4f}, f(x*) = {result_scipy.fun:.6f}")

Поиск минимума методом имитации отжига
Начальная температура: 0.01
------------------------------
Run 1: x* = 0.4182, f(x*) = 0.084212
Run 2: x* = 0.5166, f(x*) = 0.082083
Run 3: x* = 0.5215, f(x*) = 0.081981
Run 4: x* = 0.4321, f(x*) = 0.083905
Run 5: x* = 0.4585, f(x*) = 0.083326
------------------------------
Для справки (minimize_scalar): x* = 0.8835, f(x*) = 0.076314
